In [1]:
from time import sleep

import gymnasium as gym
import numpy as np
import pygame
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
from tqdm.auto import tqdm

from rl_training.agents.ppo import PPO
from rl_training.models.actor_critic import ActorCritic

/home/lbaret/rl-training/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
torch.normal(mean=torch.scalar_tensor(0.), std=torch.scalar_tensor(1.), size=(1,)).std()

/tmp/ipykernel_1759/4180841685.py:1: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  torch.normal(mean=torch.scalar_tensor(0.), std=torch.scalar_tensor(1.), size=(1,)).std()


tensor(nan)

In [2]:
T_horizon = 20
total_episodes = 1000

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
env = gym.make("CartPole-v1", render_mode=None)
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n
model = PPO(state_dim=state_dim, action_dim=action_dim, device=device)

score = 0.0
for episode_number in tqdm(
    range(total_episodes),
    total=total_episodes,
    desc="Agent is currently learning ...",
):
    s, _ = env.reset()
    done = False
    truncated = False

    while not done and not truncated:
        for t in range(T_horizon):
            prob = model.policy_old.act(s)
            a = prob[0]
            a_log_prob = prob[1]
            s_prime, r, done, truncated, info = env.step(a)
            model.put_data(
                (s, a, r / 100.0, s_prime, a_log_prob, done or truncated)
            )
            s = s_prime
            score += r
            if done or truncated:
                break
        model.update()

        score = 0.0

env.close()

env_show = gym.make("CartPole-v1", render_mode=None)
s, _ = env_show.reset()
done = False
truncated = False
total_reward = 0
while not done and not truncated:
    a, _ = model.policy_old.act(s)
    s, r, done, truncated, _ = env_show.step(a)
    total_reward += r
print(f"Score de la démonstration : {total_reward}")
env_show.close()

Agent is currently learning ...:  11%|█         | 111/1000 [00:05<00:58, 15.11it/s]/home/lbaret/rl-training/src/rl_training/agents/ppo.py:102: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  advantage = (advantage - advantage.mean()) / (advantage.std() + 1e-10)
Agent is currently learning ...:  11%|█         | 111/1000 [00:05<00:41, 21.61it/s]


ValueError: Expected parameter probs (Tensor of shape (1, 2)) of distribution Categorical(probs: torch.Size([1, 2])) to satisfy the constraint Simplex(), but found invalid values:
tensor([[nan, nan]], device='cuda:0', grad_fn=<DivBackward0>)